In [ ]:
# Run folder: outputs go to runs/07_rq1_train_german_test_english/
import os
if "REPO_ROOT" not in globals():
    REPO_ROOT = os.path.abspath("..")
RUN_DIR = os.path.join(REPO_ROOT, "runs", "07_rq1_train_german_test_english")
os.makedirs(RUN_DIR, exist_ok=True)
os.chdir(RUN_DIR)

In [ ]:
# Install requirements
%pip install -r ../../requirements.txt

In [ ]:
# Imports
import os
import json
import shutil
import re
import sys
from collections import OrderedDict

import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, ConcatDataset

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold

import torchaudio
from transformers import AutoConfig, AutoTokenizer, DebertaV2Model, Wav2Vec2FeatureExtractor, Wav2Vec2Model

import librosa
import parselmouth
from scipy.signal import find_peaks

import whisper
import glob
import matplotlib.pyplot as plt

In [ ]:
# Configuration
SOURCE_LANG = "de"
TARGET_LANG = "en"

CORPORA = {
    "de": {
        "name": "German",
        "ad_wav_dir": "../../data/RQ1/German/AD",
        "cn_wav_dir": "../../data/RQ1/German/HC",
        "ad_aug_wav_dir": "../../data/RQ1/German/AD_aug",
        "control_name": "HC",
        "whisper_language": "de",
        "cache": "de",
    },
    "en": {
        "name": "ADReSSo (train + test)",
        "ad_wav_dir": "../../data/RQ1/ADReSSo-Combined/ad",
        "cn_wav_dir": "../../data/RQ1/ADReSSo-Combined/cn",
        "control_name": "CN",
        "whisper_language": "en",
        "cache": "en",
    },
}

WHISPER_MODEL_SIZE = "medium"
TEXT_EMBEDDING_MODEL = "microsoft/mdeberta-v3-base"
WAV2VEC_MODEL = "facebook/wav2vec2-xls-r-300m"
NUM_EMBEDDING_FEATURES = AutoConfig.from_pretrained(WAV2VEC_MODEL).hidden_size
NUM_MFCC_FEATURES = 47
AUDIO_CHANNELS = 1
CUDA = True

EXTRACTION_ROOT = "./extraction"
FORCE_REEXTRACT = False

USE_AUGMENTATION = True
AUG_SUFFIX = "__aug"

CLASS_LABEL = {"ad": 1, "cn": 0, "ad_aug": 1}

def corpus_paths(lang):
    raw = os.path.join(EXTRACTION_ROOT, CORPORA[lang]["cache"])
    scaled = os.path.join("./preprocessing", CORPORA[lang]["cache"])
    p = {}
    for cls in CLASS_LABEL:
        p[f"{cls}_transcribed_dir"] = os.path.join(raw, "transcripts", cls)
        p[f"{cls}_text_dir"] = os.path.join(raw, "processed_text", cls)
        p[f"{cls}_raw_features_csv"] = os.path.join(raw, f"audio_features_{cls}_raw.csv")
        p[f"{cls}_raw_embeddings_csv"] = os.path.join(raw, f"audio_embeddings_{cls}_raw.csv")
        p[f"{cls}_features_dir"] = os.path.join(scaled, cls)
        p[f"{cls}_csv"] = os.path.join(scaled, cls, f"audio_features_{cls}_raw.csv")
        p[f"{cls}_embedding_csv"] = os.path.join(scaled, cls, f"audio_embeddings_{cls}.csv")
    return p

PATHS = {lang: corpus_paths(lang) for lang in CORPORA}

AUDIO_FEATURES_SCALER_PATH = "./preprocessing/scaler_audio_features_de.pkl"
AUDIO_EMBEDDING_SCALER_PATH = "./preprocessing/scaler_audio_emb_de.pkl"

BATCH_SIZE = 8
EPOCHS = 200
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
NUM_FOLDS = 5
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 10

FINAL_EPOCHS_OVERRIDE = None

RESULTS_DIR = "./results_final"
SAVE_MODEL_PATH = os.path.join(RESULTS_DIR, "model")
FINAL_MODEL_PATH = os.path.join(RESULTS_DIR, "final_model.pth")
LOG_PATH = os.path.join(RESULTS_DIR, "training.log")
FINAL_LOG_PATH = os.path.join(RESULTS_DIR, "final_training.log")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("./preprocessing", exist_ok=True)

print(f"RQ1: train on {CORPORA[SOURCE_LANG]['name']} -> test on {CORPORA[TARGET_LANG]['name']}")
print(f"Wav2Vec2 embedding size: {NUM_EMBEDDING_FEATURES}")

In [ ]:
# Reproducibility
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# Per-class file lists
def list_wavs(d):
    if not os.path.isdir(d):
        raise FileNotFoundError(f"{d} does not exist (check the paths in the configuration)")
    return {os.path.splitext(f)[0]: os.path.join(d, f) for f in sorted(os.listdir(d)) if f.lower().endswith('.wav')}

def read_labels_csv(csv_path, dx_ad, dx_control):
    labels = pd.read_csv(csv_path, dtype=str)
    labels.columns = labels.columns.str.strip()
    labels["ID"] = labels["ID"].str.strip().map(lambda x: os.path.splitext(x)[0] if x.lower().endswith('.wav') else x)
    labels["Dx"] = labels["Dx"].str.strip()
    unknown_dx = set(labels["Dx"]) - {dx_ad, dx_control}
    assert not unknown_dx, f"{csv_path}: unexpected Dx values {unknown_dx} (expected {dx_ad!r} / {dx_control!r})"
    n_rows = len(labels)
    labels = labels.drop_duplicates(subset=["ID", "Dx"])
    if len(labels) < n_rows:
        print(f"NOTE {os.path.basename(csv_path)}: dropped {n_rows - len(labels)} repeated rows (same ID and Dx)")
    conflicting = labels.loc[labels["ID"].duplicated(keep=False)].sort_values("ID")
    assert conflicting.empty, f"{csv_path}: IDs listed with different Dx values:\n{conflicting.to_string(index=False)}"
    return dict(zip(labels["ID"], labels["Dx"].map({dx_ad: "ad", dx_control: "cn"})))

def corpus_wav_files(cfg):
    if "labels_csv" in cfg:
        wavs = list_wavs(cfg["wav_dir"])
        id_to_cls = read_labels_csv(cfg["labels_csv"], cfg["dx_ad"], cfg["dx_control"])
        unlabeled = sorted(set(wavs) - set(id_to_cls))
        assert not unlabeled, f"{cfg['name']}: {len(unlabeled)} recordings have no label in {cfg['labels_csv']}, e.g. {unlabeled[:5]}"
        no_audio = sorted(set(id_to_cls) - set(wavs))
        if no_audio:
            print(f"WARNING {cfg['name']}: {len(no_audio)} IDs in the labels CSV have no .wav, e.g. {no_audio[:5]}")
        return {cls: [wavs[i] for i in sorted(wavs) if id_to_cls[i] == cls] for cls in ["ad", "cn"]}
    ad, cn = list_wavs(cfg["ad_wav_dir"]), list_wavs(cfg["cn_wav_dir"])
    both = sorted(set(ad) & set(cn))
    assert not both, f"{cfg['name']}: {len(both)} file names occur in both AD and {cfg['control_name']} folders, e.g. {both[:5]}"
    return {"ad": list(ad.values()), "cn": list(cn.values())}

WAV_FILES = {lang: corpus_wav_files(cfg) for lang, cfg in CORPORA.items()}

def original_id(patient_id):
    patient_id = str(patient_id)
    return patient_id.split(AUG_SUFFIX)[0] if AUG_SUFFIX in patient_id else patient_id

def augmented_wav_files(cfg, original_ad_files):
    aug = list_wavs(cfg["ad_aug_wav_dir"])
    bad = sorted(i for i in aug if AUG_SUFFIX not in i)
    assert not bad, f"{cfg['ad_aug_wav_dir']}: {len(bad)} files without '{AUG_SUFFIX}' in the name, e.g. {bad[:5]}"
    originals = {os.path.splitext(os.path.basename(f))[0] for f in original_ad_files}
    orphans = sorted(i for i in aug if original_id(i) not in originals)
    assert not orphans, (f"{cfg['ad_aug_wav_dir']}: {len(orphans)} augmented files have no original in "
                         f"{cfg['ad_wav_dir']} (they cannot be put in their original's fold), e.g. {orphans[:5]}")
    return [aug[i] for i in sorted(aug)]

for lang, cfg in CORPORA.items():
    if "ad_aug_wav_dir" not in cfg:
        continue
    if lang == SOURCE_LANG and USE_AUGMENTATION:
        WAV_FILES[lang]["ad_aug"] = augmented_wav_files(cfg, WAV_FILES[lang]["ad"])
    elif lang == SOURCE_LANG:
        print(f"NOTE {cfg['name']}: USE_AUGMENTATION = False -> {cfg['ad_aug_wav_dir']} is not used")
    else:
        print(f"NOTE {cfg['name']} is the TARGET corpus -> augmented files in {cfg['ad_aug_wav_dir']} are not used "
              f"(the test set contains original recordings only)")

for lang, cfg in CORPORA.items():
    n_ad, n_cn = len(WAV_FILES[lang]["ad"]), len(WAV_FILES[lang]["cn"])
    n_aug = len(WAV_FILES[lang].get("ad_aug", []))
    if n_ad == 0 or n_cn == 0:
        raise ValueError(f"{cfg['name']}: no recordings for {'AD' if n_ad == 0 else cfg['control_name']}")
    role = "SOURCE" if lang == SOURCE_LANG else "TARGET"
    source = f"labels from {os.path.basename(cfg['labels_csv'])}" if "labels_csv" in cfg else "labels from folders"
    aug_info = f"  + AD_aug={n_aug} (training only -> AD {n_ad + n_aug} / {cfg['control_name']} {n_cn} in training)" if n_aug else ""
    print(f"[{role}] {cfg['name']}: AD={n_ad}, {cfg['control_name']}={n_cn}, total={n_ad + n_cn}  ({source}){aug_info}")

In [ ]:
# Transcription
def transcribe_audio_files(audio_paths, model_size="base", output_dir=None, language=None):
    model = whisper.load_model(model_size)
    os.makedirs(output_dir, exist_ok=True)

    for audio_path in audio_paths:
        output_file = os.path.join(output_dir, os.path.splitext(os.path.basename(audio_path))[0] + '.txt')
        print(f"Transcribing {audio_path}...")
        result = model.transcribe(audio_path, language=language, task="transcribe")
        text = result["text"]
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"Saved transcription to {output_file}")

In [ ]:
# Acoustic features
def extract_features(audio_path, sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1):
    y, sr = librosa.load(audio_path, sr=sr)
    duration = librosa.get_duration(y=y, sr=sr)
    energy = np.array([
        sum(abs(y[i:i+frame_length]**2))
        for i in range(0, len(y), hop_length)
    ])
    threshold = np.percentile(energy, silence_threshold * 100)
    silence_frames = energy < threshold
    speech_frames = ~silence_frames
    total_speech_frames = np.sum(speech_frames)
    total_speech_time = total_speech_frames * hop_length / sr
    silence_durations = []
    current_silence = 0
    for is_silence in silence_frames:
        if is_silence:
            current_silence += 1
        elif current_silence > 0:
            silence_durations.append(current_silence * hop_length / sr)
            current_silence = 0
    num_pauses = len(silence_durations)
    total_pause_duration = sum(silence_durations)
    avg_pause_duration = np.mean(silence_durations) if silence_durations else 0
    max_pause_duration = max(silence_durations) if silence_durations else 0
    pause_duration_std = np.std(silence_durations) if silence_durations else 0
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    pitches = pitches.flatten()
    magnitudes = magnitudes.flatten()
    pitches = pitches[magnitudes > np.median(magnitudes)]
    pitch_mean = np.mean(pitches) if len(pitches) > 0 else 0
    pitch_std = np.std(pitches) if len(pitches) > 0 else 0
    pitch_range = np.ptp(pitches) if len(pitches) > 0 else 0
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    intensity = librosa.amplitude_to_db(S, ref=np.max)
    intensity_mean = np.mean(intensity)
    intensity_std = np.std(intensity)
    intensity_range = np.ptp(intensity)

    spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)[0]
    spectral_centroid_mean = np.mean(spectral_centroids)
    spectral_centroid_std = np.std(spectral_centroids)

    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)[0]
    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=frame_length, hop_length=hop_length)
    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)

    articulation_rate = total_speech_time / (duration - total_pause_duration) if (duration - total_pause_duration) > 0 else 0

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    peaks = find_peaks(onset_env, height=np.mean(onset_env))[0]
    estimated_syllables = len(peaks)
    speaking_rate = estimated_syllables / total_speech_time if total_speech_time > 0 else 0

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch(time_step=hop_length/sr)
    point_process = parselmouth.praat.call([snd, pitch], "To PointProcess (cc)")
    local_jitter = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3)
    local_shimmer = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 1.0 / 75, 1.0 / 500, 1.3, 1.6)
    harmonicity = snd.to_harmonicity_cc(time_step=hop_length/sr)
    hnr_values = harmonicity.values
    hnr_mean = np.mean(hnr_values[hnr_values != 0]) if np.any(hnr_values != 0) else 0

    formant = snd.to_formant_burg(time_step=hop_length/sr)
    f1 = []
    f2 = []
    f3 = []
    for t in np.arange(0, snd.duration, hop_length / sr):
        try:
            f1.append(formant.get_value_at_time(1, t))
            f2.append(formant.get_value_at_time(2, t))
            f3.append(formant.get_value_at_time(3, t))
        except:
            continue
    f1 = np.array(f1)
    f2 = np.array(f2)
    f3 = np.array(f3)
    formant_1_mean = np.mean(f1) if len(f1) > 0 else 0
    formant_1_std = np.std(f1) if len(f1) > 0 else 0
    formant_2_mean = np.mean(f2) if len(f2) > 0 else 0
    formant_2_std = np.std(f2) if len(f2) > 0 else 0
    formant_3_mean = np.mean(f3) if len(f3) > 0 else 0
    formant_3_std = np.std(f3) if len(f3) > 0 else 0

    features = {
        'duration': duration,
        'total_speech_time': total_speech_time,
        'speech_pause_ratio': total_speech_time / duration if duration > 0 else 0,
        'num_pauses': num_pauses,
        'total_pause_duration': total_pause_duration,
        'avg_pause_duration': avg_pause_duration,
        'max_pause_duration': max_pause_duration,
        'pause_duration_std': pause_duration_std,
        'pitch_mean': pitch_mean,
        'pitch_std': pitch_std,
        'pitch_range': pitch_range,
        'intensity_mean': intensity_mean,
        'intensity_std': intensity_std,
        'intensity_range': intensity_range,
        'articulation_rate': articulation_rate,
        'speaking_rate': speaking_rate,
        'spectral_centroid_mean': spectral_centroid_mean,
        'spectral_centroid_std': spectral_centroid_std,
        'zcr_mean': zcr_mean,
        'zcr_std': zcr_std,
        'jitter_local': local_jitter,
        'shimmer_local': local_shimmer,
        'hnr_mean': hnr_mean,
        'formant_1_mean': formant_1_mean,
        'formant_1_std': formant_1_std,
        'formant_2_mean': formant_2_mean,
        'formant_2_std': formant_2_std,
        'formant_3_mean': formant_3_mean,
        'formant_3_std': formant_3_std,
    }

    for i, (mean, std) in enumerate(zip(mfccs_mean, mfccs_std)):
        features[f'mfcc_{i+1}_mean'] = mean
        features[f'mfcc_{i+1}_std'] = std

    return features

def process_audio_feature_files(audio_paths, cls, output_csv, sr, frame_length, hop_length, silence_threshold):
    all_features = []
    for audio_path in audio_paths:
        patient_id = os.path.splitext(os.path.basename(audio_path))[0]
        print(f"Processing {audio_path}...")
        try:
            features = extract_features(
                audio_path,
                sr=sr,
                frame_length=frame_length,
                hop_length=hop_length,
                silence_threshold=silence_threshold
            )
            features['patient_id'] = patient_id
            features['class'] = cls
            all_features.append(features)
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
    df = pd.DataFrame(all_features)
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")

In [ ]:
# Wav2Vec2 embeddings
def extract_embeddings(audio_path, model, processor, device):
    speech_array, sampling_rate = torchaudio.load(audio_path)
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0)
        print(f"Converted stereo to mono for {audio_path}")
    else:
        speech_array = speech_array.squeeze(0)

    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
        speech_array = resampler(speech_array)
        sampling_rate = 16000

    speech = speech_array.numpy()

    inputs = processor(speech, sampling_rate=sampling_rate, return_tensors="pt", padding=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state
    embedding = embeddings.mean(dim=1).squeeze().cpu().numpy()

    return embedding

def process_audio_embedding_files(audio_paths, output_csv, model_name=WAV2VEC_MODEL):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name)
    model.to(device)
    model.eval()

    all_embeddings = []
    for audio_path in tqdm(audio_paths, desc=f"Embedding {os.path.dirname(output_csv)}"):
        patient_id = os.path.splitext(os.path.basename(audio_path))[0]
        try:
            embedding = extract_embeddings(audio_path, model, processor, device)
            all_embeddings.append({'patient_id': patient_id, 'embedding': embedding})
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            continue

    if len(all_embeddings) == 0:
        print("No embeddings were extracted. Please check for errors.")
        return

    df = pd.DataFrame(all_embeddings)
    embedding_cols = pd.DataFrame(df['embedding'].tolist())
    embedding_cols['patient_id'] = df['patient_id']
    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    embedding_cols.to_csv(output_csv, index=False)
    print(f"Embeddings saved to {output_csv}")

In [ ]:
# Run data extraction (both corpora)
def needs_run(output_path):
    if FORCE_REEXTRACT:
        return True
    if output_path.endswith('.csv'):
        return not os.path.exists(output_path)
    return not (os.path.isdir(output_path) and any(f.endswith('.txt') for f in os.listdir(output_path)))

for lang, cfg in CORPORA.items():
    p = PATHS[lang]
    for cls in WAV_FILES[lang]:
        wav_files = WAV_FILES[lang][cls]

        if needs_run(p[f"{cls}_transcribed_dir"]):
            transcribe_audio_files(wav_files, model_size=WHISPER_MODEL_SIZE,
                                   output_dir=p[f"{cls}_transcribed_dir"],
                                   language=cfg["whisper_language"])
        else:
            print(f"[skip] transcripts {lang}/{cls}")

        if needs_run(p[f"{cls}_raw_features_csv"]):
            process_audio_feature_files(wav_files, cls, p[f"{cls}_raw_features_csv"],
                                        sr=22050, frame_length=2048, hop_length=512, silence_threshold=0.1)
        else:
            print(f"[skip] acoustic features {lang}/{cls}")

        if needs_run(p[f"{cls}_raw_embeddings_csv"]):
            process_audio_embedding_files(wav_files, p[f"{cls}_raw_embeddings_csv"])
        else:
            print(f"[skip] wav2vec2 embeddings {lang}/{cls}")

In [ ]:
# Text cleaning
def preprocess_text(text):
    text = text.lower()
    text = ' '.join(text.split())
    text = re.sub(r'[^\w\s.,!?]', '', text)

    return text

def process_text_files(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.txt'):
            input_file = os.path.join(input_dir, filename)
            output_file = os.path.join(output_dir, filename)
            with open(input_file, 'r', encoding='utf-8') as f:
                text = f.read()
            processed_text = preprocess_text(text)
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(processed_text)

            print(f"Processed {filename}")

In [ ]:
# Audio feature scaling
def preprocess_csv(input_path, output_path, scaler_path):
    df = pd.read_csv(input_path, dtype={'patient_id': str})
    df = df.drop(['jitter_local', 'shimmer_local', 'formant_1_mean',
                  'formant_1_std', 'formant_2_mean', 'formant_2_std',
                  'formant_3_mean', 'formant_3_std', 'class'], axis=1)

    patient_ids = df['patient_id']
    df = df.drop(['patient_id'], axis=1)
    df = df.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = joblib.load(scaler_path)
    scaled_features = scaler.transform(df)
    df_scaled = pd.DataFrame(scaled_features, columns=df.columns)
    df_scaled['patient_id'] = patient_ids
    output_file = os.path.join(output_path, os.path.basename(input_path))
    df_scaled.to_csv(output_file, index=False)
    print(f"Processed file saved at: {output_file}")

In [ ]:
# Audio embedding scaling
def process_csv(input_csv, scaler_path, output_csv):
    df = pd.read_csv(input_csv, dtype={'patient_id': str})
    patient_ids = df['patient_id']
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()), axis=0)
    scaler = joblib.load(scaler_path)
    standardized_features = scaler.transform(features)
    standardized_df = pd.DataFrame(standardized_features, columns=features.columns)
    standardized_df['patient_id'] = patient_ids
    standardized_df.to_csv(output_csv, index=False)
    print(f"Processed CSV saved to '{output_csv}'")

In [ ]:
# Fit scalers on the SOURCE corpus only
AUDIO_FEATURE_DROP_COLUMNS = ['jitter_local', 'shimmer_local', 'formant_1_mean',
                              'formant_1_std', 'formant_2_mean', 'formant_2_std',
                              'formant_3_mean', 'formant_3_std', 'class']

def fit_audio_feature_scaler(raw_csv_paths, scaler_path):
    df = pd.concat([pd.read_csv(p) for p in raw_csv_paths], ignore_index=True)
    df = df.drop(AUDIO_FEATURE_DROP_COLUMNS + ['patient_id'], axis=1)
    df = df.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = StandardScaler().fit(df)
    joblib.dump(scaler, scaler_path)
    print(f"Fitted audio-feature scaler on {len(df)} recordings, {df.shape[1]} features -> {scaler_path}")

def fit_audio_embedding_scaler(raw_csv_paths, scaler_path):
    df = pd.concat([pd.read_csv(p) for p in raw_csv_paths], ignore_index=True)
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()), axis=0)
    scaler = StandardScaler().fit(features)
    joblib.dump(scaler, scaler_path)
    print(f"Fitted embedding scaler on {len(features)} recordings, {features.shape[1]} dims -> {scaler_path}")

In [ ]:
# Run data processing
src = PATHS[SOURCE_LANG]
source_classes = list(WAV_FILES[SOURCE_LANG])
fit_audio_feature_scaler([src[f"{cls}_raw_features_csv"] for cls in source_classes], AUDIO_FEATURES_SCALER_PATH)
fit_audio_embedding_scaler([src[f"{cls}_raw_embeddings_csv"] for cls in source_classes], AUDIO_EMBEDDING_SCALER_PATH)

for lang in CORPORA:
    p = PATHS[lang]
    for cls in WAV_FILES[lang]:
        if needs_run(p[f"{cls}_text_dir"]):
            process_text_files(p[f"{cls}_transcribed_dir"], p[f"{cls}_text_dir"])
        else:
            print(f"[skip] text cleaning {lang}/{cls}")

        os.makedirs(p[f"{cls}_features_dir"], exist_ok=True)
        preprocess_csv(p[f"{cls}_raw_features_csv"], p[f"{cls}_features_dir"], AUDIO_FEATURES_SCALER_PATH)
        process_csv(p[f"{cls}_raw_embeddings_csv"], AUDIO_EMBEDDING_SCALER_PATH, p[f"{cls}_embedding_csv"])

In [ ]:
# Check that all recordings were processed
for lang, cfg in CORPORA.items():
    p = PATHS[lang]
    for cls in WAV_FILES[lang]:
        cls_name = {"ad": "AD", "cn": cfg["control_name"], "ad_aug": "AD_aug"}[cls]
        wav_ids = {os.path.splitext(os.path.basename(f))[0] for f in WAV_FILES[lang][cls]}
        feat_ids = set(pd.read_csv(p[f"{cls}_csv"], dtype={'patient_id': str})['patient_id'])
        emb_ids = set(pd.read_csv(p[f"{cls}_embedding_csv"], dtype={'patient_id': str})['patient_id'])
        text_ids = {os.path.splitext(f)[0] for f in os.listdir(p[f"{cls}_text_dir"]) if f.endswith('.txt')}
        missing = {name: sorted(wav_ids - ids) for name, ids in
                   [("features", feat_ids), ("embeddings", emb_ids), ("text", text_ids)] if wav_ids - ids}
        extra = (feat_ids | emb_ids) - wav_ids
        status = "OK" if not missing else f"MISSING {({k: len(v) for k, v in missing.items()})}"
        print(f"{cfg['name']} {cls_name}: {len(wav_ids)} wavs -> {len(wav_ids & feat_ids & emb_ids & text_ids)} usable  [{status}]")
        for name, ids in missing.items():
            print(f"    no {name}: {ids[:10]}{' ...' if len(ids) > 10 else ''}")
        if extra:
            print(f"    WARNING {len(extra)} cached IDs are not in this class anymore (stale ./extraction cache?), "
                  f"e.g. {sorted(extra)[:5]} -> set FORCE_REEXTRACT = True")

In [ ]:
# Dataset
class DementiaDataset(Dataset):
    def __init__(
        self,
        audio_csv_path,
        embedding_csv_path,
        text_dir,
        label_value=None,
        tokenizer_model=TEXT_EMBEDDING_MODEL,
        max_length=512
    ):
        self.audio_data = pd.read_csv(audio_csv_path, dtype={'patient_id': str})
        self.embedding_data = pd.read_csv(embedding_csv_path, dtype={'patient_id': str})
        self.text_dir = text_dir
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        self.max_length = max_length
        self.data = pd.merge(
            self.audio_data,
            self.embedding_data,
            on='patient_id',
            suffixes=('_audio', '_embedding')
        )

        self.label_value = label_value
        self.data['label'] = self.label_value

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_row = self.data.iloc[idx]
        patient_id = data_row['patient_id']
        text_file_path = os.path.join(self.text_dir, f'{patient_id}.txt')

        with open(text_file_path, 'r', encoding='utf-8') as file:
            text = file.read()

        text_tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        audio_feature_columns = [
            col for col in self.audio_data.columns if col != 'patient_id'
        ]
        embedding_feature_columns = [
            col for col in self.embedding_data.columns if col != 'patient_id'
        ]
        audio_features = data_row[audio_feature_columns].values.astype(float)
        embedding_features = data_row[embedding_feature_columns].values.astype(float)
        label = torch.tensor(data_row['label'], dtype=torch.float32)

        audio_tensor = torch.tensor(audio_features, dtype=torch.float32)
        embedding_tensor = torch.tensor(embedding_features, dtype=torch.float32)

        return text_tokens, audio_tensor, embedding_tensor, label

def create_full_dataset(
    ad_text_dir,
    cn_text_dir,
    ad_csv,
    cn_csv,
    ad_embedding_csv,
    cn_embedding_csv
):
    ad_dataset = DementiaDataset(
        ad_csv,
        ad_embedding_csv,
        ad_text_dir,
        label_value=1
    )
    cn_dataset = DementiaDataset(
        cn_csv,
        cn_embedding_csv,
        cn_text_dir,
        label_value=0
    )
    full_dataset = ConcatDataset([ad_dataset, cn_dataset])
    return full_dataset

In [ ]:
# Model
class NeuroXVocal(nn.Module):
    def __init__(
        self,
        num_audio_features,
        num_embedding_features,
        text_embedding_model,
        hidden_size=768
    ):
        super(NeuroXVocal, self).__init__()

        self.text_model = DebertaV2Model.from_pretrained(text_embedding_model)
        self.hidden_size = self.text_model.config.hidden_size
        self.audio_fc = nn.Sequential(
            nn.Linear(num_audio_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        self.embedding_fc = nn.Sequential(
            nn.Linear(num_embedding_features, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=8,
            dropout=0.35,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.45),
            nn.Linear(self.hidden_size // 2, 1)
        )

    def forward(self, text_input, audio_input, embedding_input):
        input_ids = text_input['input_ids'].squeeze(1)
        attention_mask = text_input['attention_mask'].squeeze(1)
        text_outputs = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_embeddings = text_outputs.last_hidden_state
        audio_embeddings = self.audio_fc(audio_input)
        audio_embeddings = audio_embeddings.unsqueeze(1)
        embedding_embeddings = self.embedding_fc(embedding_input)
        embedding_embeddings = embedding_embeddings.unsqueeze(1)
        combined_embeddings = torch.cat(
            (audio_embeddings, embedding_embeddings, text_embeddings),
            dim=1
        )
        combined_embeddings = combined_embeddings.permute(1, 0, 2)
        transformer_output = self.transformer_encoder(combined_embeddings)
        pooled_output = transformer_output[0]
        logits = self.classifier(pooled_output).squeeze(-1)

        return logits

    def reset_parameters(self):
        def reset_layer_parameters(layer):
            if hasattr(layer, 'reset_parameters'):
                layer.reset_parameters()
        self.audio_fc.apply(reset_layer_parameters)
        self.embedding_fc.apply(reset_layer_parameters)
        self.transformer_encoder.apply(reset_layer_parameters)
        self.classifier.apply(reset_layer_parameters)

In [ ]:
# Training loop
def make_cv_splits(labels, groups, is_aug, n_splits=NUM_FOLDS, seed=SEED):
    labels = np.asarray(labels).astype(int)
    groups = np.asarray(groups).astype(str)
    is_aug = np.asarray(is_aug, dtype=bool)
    real = np.flatnonzero(~is_aug)
    orphans = set(groups[is_aug]) - set(groups[real])
    assert not orphans, f"augmented recordings without an original: {sorted(orphans)[:5]}"

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for _, va in skf.split(np.zeros(len(real)), labels[real]):
        val_idx = real[va]
        val_groups = np.unique(groups[val_idx])
        train_idx = np.flatnonzero(~np.isin(groups, val_groups))
        assert not np.isin(groups[train_idx], val_groups).any() and not is_aug[val_idx].any()
        splits.append((train_idx, val_idx))
    return splits

def train_model(
    build_model,
    full_dataset,
    epochs,
    learning_rate,
    log_path,
    save_model_path,
    device,
    num_folds=5,
    save_best_model=False,
    labels=None,
    splits=None
):
    criterion = BCEWithLogitsLoss()

    if labels is not None:
        labels = np.array(labels)
    if splits is not None:
        split_iter = splits
    elif labels is not None:
        kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)
        split_iter = kfold.split(np.zeros(len(labels)), labels)
    else:
        kfold = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
        split_iter = kfold.split(full_dataset)

    history = {}
    overall_best_val_loss = float('inf')
    overall_best_fold = None

    for fold, (train_indices, val_indices) in enumerate(split_iter):
        print(f'Fold {fold+1}/{num_folds}')

        if labels is not None:
            train_counts = np.bincount(labels[train_indices].astype(int), minlength=2)
            val_counts = np.bincount(labels[val_indices].astype(int), minlength=2)
            print(f"Fold {fold+1}: train CN/AD = {train_counts[0]}/{train_counts[1]}, val CN/AD = {val_counts[0]}/{val_counts[1]}")

        train_subset = torch.utils.data.Subset(full_dataset, train_indices)
        val_subset = torch.utils.data.Subset(full_dataset, val_indices)

        train_loader = torch.utils.data.DataLoader(
            train_subset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED)
        )
        val_loader = torch.utils.data.DataLoader(
            val_subset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = build_model()
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model.to(device)

        optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=1e-4
        )
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5
        )

        best_val_loss = float('inf')
        epochs_no_improve = 0
        history[fold] = {'train_loss': [], 'val_loss': [], 'lr': [], 'best_epoch': None}

        for epoch in range(epochs):
            history[fold]['lr'].append(optimizer.param_groups[0]['lr'])
            model.train()
            running_loss = 0.0
            all_train_outputs = []
            all_train_labels = []

            for (
                text_data,
                audio_data,
                embedding_data,
                label
            ) in tqdm(
                train_loader,
                desc=f"Fold {fold+1}, Epoch {epoch+1}/{epochs} - Training"
            ):
                optimizer.zero_grad()
                text_data = {
                    key: value.to(device) for key, value in text_data.items()
                }
                audio_data = audio_data.to(device)
                embedding_data = embedding_data.to(device)
                label = label.to(device)

                outputs = model(text_data, audio_data, embedding_data)

                loss = criterion(outputs, label.float())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                all_train_outputs.extend(outputs.detach().cpu().numpy())
                all_train_labels.extend(label.cpu().numpy())

            train_loss = running_loss / len(train_loader)
            train_predictions = (np.array(all_train_outputs) > 0).astype(int)
            train_accuracy = accuracy_score(
                np.array(all_train_labels),
                train_predictions
            )
            train_report = classification_report(
                np.array(all_train_labels),
                train_predictions,
                target_names=['Control', 'ProbableAD'],
                digits=4
            )

            model.eval()
            val_running_loss = 0.0
            all_val_outputs = []
            all_val_labels = []
            with torch.no_grad():
                for (
                    text_data,
                    audio_data,
                    embedding_data,
                    label
                ) in tqdm(
                    val_loader,
                    desc=f"Fold {fold+1}, Epoch {epoch+1}/{epochs} - Validation"
                ):
                    text_data = {
                        key: value.to(device) for key, value in text_data.items()
                    }
                    audio_data = audio_data.to(device)
                    embedding_data = embedding_data.to(device)
                    label = label.to(device)

                    outputs = model(text_data, audio_data, embedding_data)
                    loss = criterion(outputs, label.float())

                    val_running_loss += loss.item()
                    all_val_outputs.extend(outputs.detach().cpu().numpy())
                    all_val_labels.extend(label.cpu().numpy())

            val_loss = val_running_loss / len(val_loader)
            val_predictions = (np.array(all_val_outputs) > 0).astype(int)
            val_accuracy = accuracy_score(
                np.array(all_val_labels),
                val_predictions
            )
            val_report = classification_report(
                np.array(all_val_labels),
                val_predictions,
                target_names=['Control', 'ProbableAD'],
                digits=4
            )

            history[fold]['train_loss'].append(train_loss)
            history[fold]['val_loss'].append(val_loss)

            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                history[fold]['best_epoch'] = epoch + 1
                if save_best_model:
                    best_model_path = f"{save_model_path}_fold{fold+1}_best.pth"
                    if isinstance(model, torch.nn.DataParallel):
                        torch.save(model.module.state_dict(), best_model_path)
                    else:
                        torch.save(model.state_dict(), best_model_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping at epoch {epoch+1} for fold {fold+1}")
                    break

            with open(log_path, 'a') as f:
                f.write(
                    f"Fold {fold+1}, Epoch {epoch+1}, "
                    f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}\n"
                )
                f.write("Train Classification Report:\n")
                f.write(f"{train_report}\n")
                f.write("Validation Classification Report:\n")
                f.write(f"{val_report}\n")

            print(
                f"Fold {fold+1}, Epoch {epoch+1}, "
                f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}"
            )
            print("Train Classification Report:")
            print(train_report)
            print("Validation Classification Report:")
            print(val_report)

        print(f"Fold {fold+1}: best epoch {history[fold]['best_epoch']} (val loss {best_val_loss:.4f})")

        if best_val_loss < overall_best_val_loss:
            overall_best_val_loss = best_val_loss
            overall_best_fold = fold

        del model, optimizer, scheduler
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if save_best_model and overall_best_fold is not None:
        best_fold_checkpoint = f"{save_model_path}_fold{overall_best_fold+1}_best.pth"
        overall_best_checkpoint = f"{save_model_path}_best.pth"
        shutil.copy2(best_fold_checkpoint, overall_best_checkpoint)
        print(f"Overall best fold: {overall_best_fold+1} (val loss {overall_best_val_loss:.4f}) -> saved as {overall_best_checkpoint}")

    return history

In [ ]:
# Run training
print(f"Starting k-fold cross-validation training on SOURCE = {CORPORA[SOURCE_LANG]['name']}...")
with open(LOG_PATH, 'w') as f:
    pass

src = PATHS[SOURCE_LANG]
source_classes = list(WAV_FILES[SOURCE_LANG])
train_parts = [DementiaDataset(src[f"{cls}_csv"], src[f"{cls}_embedding_csv"], src[f"{cls}_text_dir"],
                               label_value=CLASS_LABEL[cls]) for cls in source_classes]
full_dataset = ConcatDataset(train_parts)

device = torch.device('cuda' if torch.cuda.is_available() and CUDA else 'cpu')

def build_model():
    return NeuroXVocal(
        num_audio_features=NUM_MFCC_FEATURES,
        num_embedding_features=NUM_EMBEDDING_FEATURES,
        text_embedding_model=TEXT_EMBEDDING_MODEL
    )

labels = np.concatenate([ds.data['label'].values for ds in train_parts]).astype(int)
train_patient_ids = np.concatenate([ds.data['patient_id'].astype(str).values for ds in train_parts])
is_aug = np.concatenate([np.full(len(ds), cls == "ad_aug") for cls, ds in zip(source_classes, train_parts)])
groups = np.array([original_id(i) for i in train_patient_ids])
assert not is_aug[:int((~is_aug).sum())].any(), "augmented rows must come after all original rows"
assert all((AUG_SUFFIX in i) == a for i, a in zip(train_patient_ids, is_aug)), "AUG_SUFFIX in an original ID?"

cv_splits = make_cv_splits(labels, groups, is_aug)

val_fold_of_group = {g: k + 1 for k, (_, va) in enumerate(cv_splits) for g in groups[va]}
cv_folds_df = pd.DataFrame({'patient_id': train_patient_ids, 'group': groups, 'label': labels,
                            'augmented': is_aug, 'val_fold': [val_fold_of_group[g] for g in groups]})
cv_folds_df.to_csv(os.path.join(RESULTS_DIR, "cv_folds.csv"), index=False)

print(f"{len(full_dataset)} source rows: {int((~is_aug).sum())} original + {int(is_aug.sum())} augmented")
for k, (tr, va) in enumerate(cv_splits):
    print(f"Fold {k+1}: train {len(tr)} (of which {int(is_aug[tr].sum())} augmented, AD/control "
          f"{int(labels[tr].sum())}/{int((labels[tr] == 0).sum())}) | val {len(va)} originals "
          f"(AD/control {int(labels[va].sum())}/{int((labels[va] == 0).sum())}) | "
          f"{int((is_aug & np.isin(groups, groups[va])).sum())} augmented copies of val recordings left out")

history = train_model(
    build_model,
    full_dataset,
    EPOCHS,
    LEARNING_RATE,
    LOG_PATH,
    SAVE_MODEL_PATH,
    device,
    NUM_FOLDS,
    SAVE_BEST_MODEL,
    labels=labels,
    splits=cv_splits
)

In [ ]:
# Plot training and validation loss
n_folds = len(history)
fig, axes = plt.subplots(1, n_folds, figsize=(5 * n_folds, 4), sharey=True)
if n_folds == 1:
    axes = [axes]

for ax, fold in zip(axes, sorted(history.keys())):
    epochs_range = range(1, len(history[fold]['train_loss']) + 1)
    ax.plot(epochs_range, history[fold]['train_loss'], label='Train Loss')
    ax.plot(epochs_range, history[fold]['val_loss'], label='Val Loss')
    ax.set_title(f'Fold {fold + 1}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
per_fold_plot_path = os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "loss_curves_per_fold.png")
plt.savefig(per_fold_plot_path)
plt.show()
print(f"Saved plot to {per_fold_plot_path}")

In [ ]:
# Plot training and validation loss per fold
plt.figure(figsize=(8, 5))
for fold in sorted(history.keys()):
    epochs_range = range(1, len(history[fold]['train_loss']) + 1)
    plt.plot(epochs_range, history[fold]['train_loss'], linestyle='-', label=f'Fold {fold + 1} Train')
    plt.plot(epochs_range, history[fold]['val_loss'], linestyle='--', label=f'Fold {fold + 1} Val')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss (All Folds)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
overview_plot_path = os.path.join(os.path.dirname(SAVE_MODEL_PATH) or ".", "loss_curves_all_folds.png")
plt.savefig(overview_plot_path)
plt.show()
print(f"Saved plot to {overview_plot_path}")

In [ ]:
# Stage B recipe
fold_ids = sorted(history.keys())
best_epochs = [history[f]['best_epoch'] for f in fold_ids]
cv_final_epochs = int(np.round(np.median(best_epochs)))
FINAL_EPOCHS = FINAL_EPOCHS_OVERRIDE if FINAL_EPOCHS_OVERRIDE is not None else cv_final_epochs

def cv_lr_at(epoch_idx):
    return float(np.median([history[f]['lr'][min(epoch_idx, len(history[f]['lr']) - 1)] for f in fold_ids]))

FINAL_LR_SCHEDULE = [cv_lr_at(e) for e in range(FINAL_EPOCHS)]

recipe_df = pd.DataFrame({
    'fold': [f + 1 for f in fold_ids],
    'best_epoch': best_epochs,
    'epochs_run': [len(history[f]['train_loss']) for f in fold_ids],
    'best_val_loss': [min(history[f]['val_loss']) for f in fold_ids],
    'lr_at_best_epoch': [history[f]['lr'][history[f]['best_epoch'] - 1] for f in fold_ids],
})
print(recipe_df.to_string(index=False))
print()
print(f"Median best epoch: {cv_final_epochs}  ->  FINAL_EPOCHS = {FINAL_EPOCHS}"
      + ("  (override)" if FINAL_EPOCHS_OVERRIDE is not None else ""))
lr_changes = [e + 1 for e in range(1, FINAL_EPOCHS) if FINAL_LR_SCHEDULE[e] != FINAL_LR_SCHEDULE[e - 1]]
print(f"Final LR schedule: starts at {FINAL_LR_SCHEDULE[0]:.2e}, ends at {FINAL_LR_SCHEDULE[-1]:.2e}, "
      f"changes at epochs {lr_changes if lr_changes else 'none (constant)'}")

recipe = {
    'source': CORPORA[SOURCE_LANG]['name'],
    'cv_best_epochs': best_epochs,
    'cv_median_best_epoch': cv_final_epochs,
    'final_epochs': FINAL_EPOCHS,
    'final_epochs_override': FINAL_EPOCHS_OVERRIDE,
    'final_lr_schedule': FINAL_LR_SCHEDULE,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
}
recipe_path = os.path.join(RESULTS_DIR, "final_training_recipe.json")
with open(recipe_path, 'w') as f:
    json.dump(recipe, f, indent=2)
print(f"Saved {recipe_path}")

In [ ]:
# Stage B — final training on 100% of the source corpus
def train_final_model(build_model, dataset, lr_schedule, log_path, save_path, device):
    criterion = BCEWithLogitsLoss()

    torch.manual_seed(SEED)
    model = build_model()
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr_schedule[0], weight_decay=1e-4)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )

    history = {'train_loss': [], 'train_accuracy': [], 'lr': []}
    with open(log_path, 'w') as f:
        pass

    n_epochs = len(lr_schedule)
    for epoch, lr in enumerate(lr_schedule):
        for group in optimizer.param_groups:
            group['lr'] = lr

        model.train()
        running_loss = 0.0
        all_outputs, all_labels = [], []
        for text_data, audio_data, embedding_data, label in tqdm(
            loader, desc=f"Final model, Epoch {epoch+1}/{n_epochs}"
        ):
            optimizer.zero_grad()
            text_data = {key: value.to(device) for key, value in text_data.items()}
            audio_data = audio_data.to(device)
            embedding_data = embedding_data.to(device)
            label = label.to(device)

            outputs = model(text_data, audio_data, embedding_data)
            loss = criterion(outputs, label.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            all_outputs.extend(outputs.detach().cpu().numpy())
            all_labels.extend(label.cpu().numpy())

        train_loss = running_loss / len(loader)
        train_predictions = (np.array(all_outputs) > 0).astype(int)
        train_accuracy = accuracy_score(np.array(all_labels), train_predictions)
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['lr'].append(lr)

        msg = f"Final model, Epoch {epoch+1}/{n_epochs}, LR: {lr:.2e}, Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}"
        print(msg)
        with open(log_path, 'a') as f:
            f.write(msg + "\n")

    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state_dict, save_path)
    print(f"Saved final model (after epoch {n_epochs}) to {save_path}")

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return history

n_ad, n_cn = int(np.sum(labels)), int(len(labels) - np.sum(labels))
print(f"Final training on 100% of {CORPORA[SOURCE_LANG]['name']}: {len(full_dataset)} recordings "
      f"(AD {n_ad} / {CORPORA[SOURCE_LANG]['control_name']} {n_cn}; {int(is_aug.sum())} of the AD rows are "
      f"augmented copies), {FINAL_EPOCHS} epochs")

final_history = train_final_model(
    build_model,
    full_dataset,
    FINAL_LR_SCHEDULE,
    FINAL_LOG_PATH,
    FINAL_MODEL_PATH,
    device
)

In [ ]:
# Plot the training curve of the final model
fig, ax1 = plt.subplots(figsize=(8, 4))
epochs_range = range(1, len(final_history['train_loss']) + 1)
ax1.plot(epochs_range, final_history['train_loss'], label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax2 = ax1.twinx()
ax2.plot(epochs_range, final_history['train_accuracy'], color='tab:orange', linestyle='--', label='Train Accuracy')
ax2.set_ylabel('Accuracy')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
ax1.set_title(f"Final model on 100% of {CORPORA[SOURCE_LANG]['name']} (no validation split)")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
final_plot_path = os.path.join(RESULTS_DIR, "final_model_training_curve.png")
plt.savefig(final_plot_path)
plt.show()
print(f"Saved plot to {final_plot_path}")

In [ ]:
# Evaluation helpers
class LabeledPatientDataset(DementiaDataset):
    def __getitem__(self, idx):
        text_tokens, audio_tensor, embedding_tensor, label = super().__getitem__(idx)
        return text_tokens, audio_tensor, embedding_tensor, label, self.data.iloc[idx]['patient_id']

def create_labeled_dataset(paths):
    return ConcatDataset([
        LabeledPatientDataset(paths["ad_csv"], paths["ad_embedding_csv"], paths["ad_text_dir"], label_value=1),
        LabeledPatientDataset(paths["cn_csv"], paths["cn_embedding_csv"], paths["cn_text_dir"], label_value=0),
    ])

def load_checkpoint(checkpoint_path, device):
    model = NeuroXVocal(
        num_audio_features=NUM_MFCC_FEATURES,
        num_embedding_features=NUM_EMBEDDING_FEATURES,
        text_embedding_model=TEXT_EMBEDDING_MODEL,
    )
    state_dict = torch.load(checkpoint_path, map_location=device)
    if 'module.' in list(state_dict.keys())[0]:
        state_dict = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

def predict(model, dataset, device, batch_size=BATCH_SIZE):
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    probs, labels_, ids = [], [], []
    with torch.no_grad():
        for text_data, audio_data, embedding_data, label, patient_id in loader:
            text_data = {key: value.to(device) for key, value in text_data.items()}
            logits = model(text_data, audio_data.to(device), embedding_data.to(device))
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            labels_.extend(label.numpy().astype(int).tolist())
            ids.extend(list(patient_id))
    return np.array(ids), np.array(labels_), np.array(probs)

def compute_metrics(y_true, probs, threshold=0.5):
    y_pred = (probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'n': len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_ad': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'sensitivity': tp / (tp + fn) if (tp + fn) else float('nan'),
        'specificity': tn / (tn + fp) if (tn + fp) else float('nan'),
        'roc_auc': roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else float('nan'),
        'pred_ad_rate': y_pred.mean(),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
    }

def fold_checkpoints():
    paths = [f"{SAVE_MODEL_PATH}_fold{k + 1}_best.pth" for k in range(NUM_FOLDS)]
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Missing fold checkpoints: {missing}")
    return paths

def freeze_model(model):
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)
    return model

device = torch.device('cuda' if torch.cuda.is_available() and CUDA else 'cpu')

CV_FOLDS_CSV = os.path.join(RESULTS_DIR, "cv_folds.csv")

def source_cv_splits(dataset):
    labels_ = np.concatenate([ds.data['label'].values for ds in dataset.datasets]).astype(int)
    ids_ = np.concatenate([ds.data['patient_id'].astype(str).values for ds in dataset.datasets])
    splits = make_cv_splits(labels_, [original_id(i) for i in ids_], np.zeros(len(ids_), dtype=bool))
    if os.path.exists(CV_FOLDS_CSV):
        folds = pd.read_csv(CV_FOLDS_CSV, dtype={'patient_id': str})
        for k, (_, va) in enumerate(splits):
            trained = set(folds.loc[(folds['val_fold'] == k + 1) & ~folds['augmented'], 'patient_id'])
            assert set(ids_[va]) == trained, f"fold {k+1}: validation split differs from training"
    else:
        print(f"WARNING {CV_FOLDS_CSV} not found - cannot check the folds against training")
    return splits

In [ ]:
# In-domain reference: source out-of-fold predictions
source_dataset = create_labeled_dataset(PATHS[SOURCE_LANG])
source_labels = np.concatenate([ds.data['label'].values for ds in source_dataset.datasets]).astype(int)

oof_frames, source_fold_rows = [], []
for fold, ((_, val_indices), ckpt) in enumerate(zip(source_cv_splits(source_dataset), fold_checkpoints())):
    model = load_checkpoint(ckpt, device)
    ids, y, p = predict(model, torch.utils.data.Subset(source_dataset, val_indices), device)
    oof_frames.append(pd.DataFrame({'patient_id': ids, 'fold': fold + 1, 'true_label': y, 'predicted_probability': p}))
    source_fold_rows.append({'fold': fold + 1, **compute_metrics(y, p)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

source_oof_df = pd.concat(oof_frames, ignore_index=True)
source_oof_df['predicted_label'] = (source_oof_df['predicted_probability'] > 0.5).astype(int)
source_fold_df = pd.DataFrame(source_fold_rows)
source_oof_metrics = compute_metrics(source_oof_df['true_label'].values, source_oof_df['predicted_probability'].values)

source_oof_df.to_csv(os.path.join(RESULTS_DIR, f"source_{SOURCE_LANG}_oof_predictions.csv"), index=False)
source_fold_df.to_csv(os.path.join(RESULTS_DIR, f"source_{SOURCE_LANG}_per_fold_metrics.csv"), index=False)

print(f"In-domain ({CORPORA[SOURCE_LANG]['name']}) out-of-fold, pooled over {NUM_FOLDS} folds:")
print(pd.Series(source_oof_metrics).to_string())
source_fold_df

In [ ]:
# Cross-lingual transfer (RQ1 answer)
if not os.path.exists(FINAL_MODEL_PATH):
    raise FileNotFoundError(f"{FINAL_MODEL_PATH} not found - train the final model first")

target_dataset = create_labeled_dataset(PATHS[TARGET_LANG])

final_model = freeze_model(load_checkpoint(FINAL_MODEL_PATH, device))
n_trainable = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
assert n_trainable == 0 and not final_model.training, "final model is not frozen"
print(f"Final model frozen: {sum(p.numel() for p in final_model.parameters()):,} parameters, 0 trainable, eval mode")

target_ids, target_labels, final_probs = predict(final_model, target_dataset, device)
final_target_metrics = compute_metrics(target_labels, final_probs)
del final_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_pred_df = pd.DataFrame({
    'patient_id': target_ids,
    'true_label': target_labels,
    'predicted_probability': final_probs,
    'predicted_label': (final_probs > 0.5).astype(int),
})
final_pred_path = os.path.join(RESULTS_DIR, f"target_{TARGET_LANG}_final_model_predictions.csv")
final_pred_df.to_csv(final_pred_path, index=False)

print(f"Cross-lingual ({CORPORA[SOURCE_LANG]['name']} -> {CORPORA[TARGET_LANG]['name']}), "
      f"final model trained on 100% of the source corpus, frozen:")
print(pd.Series(final_target_metrics).to_string())
print()
print(classification_report(target_labels, (final_probs > 0.5).astype(int),
                            target_names=['Control', 'ProbableAD'], digits=4))
print(f"Saved {final_pred_path}")

In [ ]:
# Fold models on the target corpus (comparison only)
target_fold_rows, target_fold_probs = [], []
for fold, ckpt in enumerate(fold_checkpoints()):
    model = load_checkpoint(ckpt, device)
    ids, y, p = predict(model, target_dataset, device)
    assert (ids == target_ids).all() and (y == target_labels).all()
    target_fold_probs.append(p)
    target_fold_rows.append({'fold': fold + 1, **compute_metrics(y, p)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

target_fold_df = pd.DataFrame(target_fold_rows)
ensemble_probs = np.mean(np.array(target_fold_probs), axis=0)
target_ensemble_metrics = compute_metrics(target_labels, ensemble_probs)

target_pred_df = pd.DataFrame({'patient_id': target_ids, 'true_label': target_labels})
for k, p in enumerate(target_fold_probs):
    target_pred_df[f'prob_fold{k + 1}'] = p
target_pred_df['predicted_probability'] = ensemble_probs
target_pred_df['predicted_label'] = (ensemble_probs > 0.5).astype(int)

target_pred_df.to_csv(os.path.join(RESULTS_DIR, f"target_{TARGET_LANG}_cv_ensemble_predictions.csv"), index=False)
target_fold_df.to_csv(os.path.join(RESULTS_DIR, f"target_{TARGET_LANG}_cv_per_fold_metrics.csv"), index=False)

print(f"Cross-lingual ({CORPORA[SOURCE_LANG]['name']} -> {CORPORA[TARGET_LANG]['name']}), "
      f"CV-fold ensemble of {len(target_fold_probs)} folds (comparison only):")
print(pd.Series(target_ensemble_metrics).to_string())
print()
print(classification_report(target_labels, (ensemble_probs > 0.5).astype(int),
                            target_names=['Control', 'ProbableAD'], digits=4))
target_fold_df

In [ ]:
# RQ1 summary for this direction
metric_cols = ['accuracy', 'balanced_accuracy', 'f1_ad', 'sensitivity', 'specificity', 'roc_auc', 'pred_ad_rate']

oof_row = f'in-domain {SOURCE_LANG} (OOF pooled)'
final_row = f'{SOURCE_LANG}->{TARGET_LANG} final model (frozen)'
summary_rows = [
    {'setting': oof_row, **{m: source_oof_metrics[m] for m in metric_cols}},
    {'setting': f'in-domain {SOURCE_LANG} (per-fold mean)', **source_fold_df[metric_cols].mean().to_dict()},
    {'setting': f'in-domain {SOURCE_LANG} (per-fold std)', **source_fold_df[metric_cols].std().to_dict()},
    {'setting': final_row, **{m: final_target_metrics[m] for m in metric_cols}},
    {'setting': f'{SOURCE_LANG}->{TARGET_LANG} CV-fold ensemble (comparison)', **{m: target_ensemble_metrics[m] for m in metric_cols}},
    {'setting': f'{SOURCE_LANG}->{TARGET_LANG} CV per-fold mean (comparison)', **target_fold_df[metric_cols].mean().to_dict()},
    {'setting': f'{SOURCE_LANG}->{TARGET_LANG} CV per-fold std (comparison)', **target_fold_df[metric_cols].std().to_dict()},
]
summary_df = pd.DataFrame(summary_rows).set_index('setting')
summary_df.loc['transfer gap (OOF - final model)'] = summary_df.loc[oof_row] - summary_df.loc[final_row]

print(f"Final model: {FINAL_EPOCHS} epochs on 100% of {CORPORA[SOURCE_LANG]['name']} (CV best epochs: {best_epochs})")
print(f"Target true AD share: {target_labels.mean():.3f}  |  source true AD share: {source_labels.mean():.3f}")
summary_path = os.path.join(RESULTS_DIR, f"rq1_summary_{SOURCE_LANG}_to_{TARGET_LANG}.csv")
summary_df.to_csv(summary_path)
print(f"Saved {summary_path}")
summary_df.round(4)

In [ ]:
# Independent test results (target corpus)
test_predictions = (final_probs > 0.5).astype(int)
test_accuracy = accuracy_score(target_labels, test_predictions)
test_report = classification_report(
    target_labels, test_predictions, target_names=['Control', 'ProbableAD'], digits=4
)

m = final_target_metrics
majority_baseline = max(target_labels.mean(), 1 - target_labels.mean())
test_summary = (
    f"Independent test on {CORPORA[TARGET_LANG]['name']} (final model, {FINAL_EPOCHS} epochs on 100% of {CORPORA[SOURCE_LANG]['name']}, frozen, n={m['n']}):\n"
    f"  ROC-AUC           : {m['roc_auc']:.4f}\n"
    f"  Balanced accuracy : {m['balanced_accuracy']:.4f}\n"
    f"  Sensitivity (AD)  : {m['sensitivity']:.4f}\n"
    f"  Specificity (Ctrl): {m['specificity']:.4f}\n"
    f"  F1 (AD)           : {m['f1_ad']:.4f}\n"
    f"  Accuracy          : {test_accuracy:.4f}  (majority-class baseline: {majority_baseline:.4f})\n"
    f"  Confusion matrix  : TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}\n"
)

print(test_summary)
print(test_report)

test_results_df = pd.DataFrame({
    'patient_id': target_ids,
    'true_label': target_labels,
    'predicted_probability': final_probs,
    'predicted_label': test_predictions,
})

results_csv_path = os.path.join(RESULTS_DIR, "test_evaluation.csv")
report_path = os.path.join(RESULTS_DIR, "test_report.txt")
test_results_df.to_csv(results_csv_path, index=False)
with open(report_path, 'w') as f:
    f.write(test_summary + "\n")
    f.write(test_report)
print(f"Saved per-patient results to {results_csv_path}")
print(f"Saved metrics/classification report to {report_path}")
test_results_df

In [ ]:
# Transferability estimates: LEEP, NCE and LogME
LOGME_REPO = None
LOGME_GIT_URL = "https://github.com/thuml/LogME.git"

def find_logme_repo():
    if LOGME_REPO:
        return os.path.abspath(LOGME_REPO)
    here = os.path.abspath(os.getcwd())
    while True:
        candidate = os.path.join(here, "../../LogME")
        if os.path.isfile(os.path.join(candidate, "LogME.py")):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

LOGME_REPO = find_logme_repo()
if LOGME_REPO is None:
    LOGME_REPO = os.path.abspath(os.path.join("..", "..", "../../LogME"))
    print(f"LogME repo not found -> cloning {LOGME_GIT_URL} into {LOGME_REPO}")
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", LOGME_GIT_URL, LOGME_REPO], check=True)
if not os.path.isfile(os.path.join(LOGME_REPO, "LogME.py")):
    raise FileNotFoundError(f"LogME.py not found in {LOGME_REPO}. Copy the LogME folder from the project root "
                            f"to this machine, or set LOGME_REPO to the folder that contains LogME.py")
print(f"Using LogME repo: {LOGME_REPO}")
if LOGME_REPO not in sys.path:
    sys.path.insert(0, LOGME_REPO)
from LEEP import LEEP
from NCE import NCE
from LogME import LogME

def predict_with_features(model, dataset, device, batch_size=BATCH_SIZE):
    features = []
    hook = model.classifier.register_forward_hook(
        lambda module, inputs, output: features.append(inputs[0].detach().cpu().numpy()))
    try:
        ids, y, p = predict(model, dataset, device, batch_size)
    finally:
        hook.remove()
    return ids, y, p, np.concatenate(features).astype(np.float64)

def label_entropy(y):
    prior = np.bincount(np.asarray(y, dtype=int), minlength=2) / len(y)
    prior = prior[prior > 0]
    return float(-(prior * np.log(prior)).sum())

def transferability_scores(y, probs, features=None, threshold=0.5):
    y = np.asarray(y, dtype=int)
    probs = np.asarray(probs, dtype=np.float64)
    dummy_dist = np.stack([1.0 - probs, probs], axis=1)
    dummy_label = (probs > threshold).astype(int)
    h_y = label_entropy(y)
    leep = float(LEEP(dummy_dist, y))
    nce = float(NCE(dummy_label, y))
    logme = float(LogME(regression=False).fit(features, y)) if features is not None else float('nan')
    return {
        'leep': leep,
        'nce': nce,
        'logme': logme,
        'label_entropy': h_y,
        'leep_minus_prior': leep + h_y,
        'nce_normalized': 1.0 + nce / h_y if h_y > 0 else float('nan'),
    }

TRANSFER_COLS = ['leep', 'nce', 'logme', 'label_entropy', 'leep_minus_prior', 'nce_normalized']

In [ ]:
# Transferability estimates: LEEP, NCE and LogME (continued)
per_fold_rows = []

oof_y, oof_p = [], []
for fold, ((_, val_indices), ckpt) in enumerate(zip(source_cv_splits(source_dataset), fold_checkpoints())):
    model = freeze_model(load_checkpoint(ckpt, device))
    _, y, p, f = predict_with_features(model, torch.utils.data.Subset(source_dataset, val_indices), device)
    oof_y.append(y)
    oof_p.append(p)
    per_fold_rows.append({'evaluated_on': f'source {SOURCE_LANG} (val split)', 'fold': fold + 1, 'n': len(y),
                          **transferability_scores(y, p, f)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
source_oof_scores = transferability_scores(np.concatenate(oof_y), np.concatenate(oof_p))

final_model = freeze_model(load_checkpoint(FINAL_MODEL_PATH, device))
t_ids, t_y, t_p, target_features = predict_with_features(final_model, target_dataset, device)
del final_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
assert (t_ids == target_ids).all() and (t_y == target_labels).all()
assert np.allclose(t_p, final_probs, atol=1e-5), "predictions differ from the earlier evaluation"
final_scores = transferability_scores(t_y, t_p, target_features)
features_path = os.path.join(RESULTS_DIR, f"target_{TARGET_LANG}_final_model_features.npy")
np.save(features_path, target_features)

fold_target_probs = []
for fold, ckpt in enumerate(fold_checkpoints()):
    model = freeze_model(load_checkpoint(ckpt, device))
    ids, y, p, f = predict_with_features(model, target_dataset, device)
    assert (ids == target_ids).all()
    fold_target_probs.append(p)
    per_fold_rows.append({'evaluated_on': f'target {TARGET_LANG}', 'fold': fold + 1, 'n': len(y),
                          **transferability_scores(y, p, f)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
ensemble_scores = transferability_scores(target_labels, np.mean(np.array(fold_target_probs), axis=0))

per_fold_transfer_df = pd.DataFrame(per_fold_rows)
src_folds = per_fold_transfer_df[per_fold_transfer_df['evaluated_on'].str.startswith('source')][TRANSFER_COLS]
tgt_folds = per_fold_transfer_df[per_fold_transfer_df['evaluated_on'].str.startswith('target')][TRANSFER_COLS]

transfer_df = pd.DataFrame.from_dict({
    oof_row: source_oof_scores,
    f'in-domain {SOURCE_LANG} (per-fold mean)': src_folds.mean().to_dict(),
    f'in-domain {SOURCE_LANG} (per-fold std)': src_folds.std().to_dict(),
    final_row: final_scores,
    f'{SOURCE_LANG}->{TARGET_LANG} CV-fold ensemble (comparison)': ensemble_scores,
    f'{SOURCE_LANG}->{TARGET_LANG} CV per-fold mean (comparison)': tgt_folds.mean().to_dict(),
    f'{SOURCE_LANG}->{TARGET_LANG} CV per-fold std (comparison)': tgt_folds.std().to_dict(),
}, orient='index')[TRANSFER_COLS]
transfer_df.index.name = 'setting'
transfer_df.loc['transfer gap (OOF - final model)'] = transfer_df.loc[oof_row] - transfer_df.loc[final_row]

transfer_path = os.path.join(RESULTS_DIR, f"transferability_scores_{SOURCE_LANG}_to_{TARGET_LANG}.csv")
per_fold_path = os.path.join(RESULTS_DIR, f"transferability_per_fold_{SOURCE_LANG}_to_{TARGET_LANG}.csv")
transfer_df.to_csv(transfer_path)
per_fold_transfer_df.to_csv(per_fold_path, index=False)

print(f"Transferability {CORPORA[SOURCE_LANG]['name']} -> {CORPORA[TARGET_LANG]['name']} "
      f"(chance level: LEEP = NCE = -H(Y); nce_normalized = leep_minus_prior = 0)")
print(f"Target H(Y) = {final_scores['label_entropy']:.4f}  |  source H(Y) = {source_oof_scores['label_entropy']:.4f}  "
      f"|  LogME features: {target_features.shape[0]} recordings x {target_features.shape[1]} dims")
print(f"Saved {transfer_path}\nSaved {per_fold_path}\nSaved {features_path}")
transfer_df.round(4)

In [ ]:
# Transferability estimates: LEEP, NCE and LogME (continued)
summary_df = summary_df.drop(columns=[c for c in TRANSFER_COLS if c in summary_df.columns]).join(transfer_df)
summary_df.to_csv(summary_path)
print(f"Updated {summary_path} with {TRANSFER_COLS}")

f = final_scores
transfer_summary = (
    f"Transferability of the frozen final model to {CORPORA[TARGET_LANG]['name']} (n={len(target_labels)}):\n"
    f"  LEEP              : {f['leep']:.4f}  (chance -H(Y) = {-f['label_entropy']:.4f}, gain over prior {f['leep_minus_prior']:.4f})\n"
    f"  NCE               : {f['nce']:.4f}  (chance -H(Y) = {-f['label_entropy']:.4f}, normalized {f['nce_normalized']:.4f})\n"
    f"  LogME             : {f['logme']:.4f}  (CV fold models on target: "
    f"{tgt_folds['logme'].mean():.4f} +/- {tgt_folds['logme'].std():.4f})\n"
    f"  In-domain {SOURCE_LANG} OOF  : LEEP {source_oof_scores['leep']:.4f}, NCE {source_oof_scores['nce']:.4f}, "
    f"normalized NCE {source_oof_scores['nce_normalized']:.4f}\n"
)
print(test_summary)
print(transfer_summary)
with open(report_path, 'w') as fh:
    fh.write(test_summary + "\n")
    fh.write(transfer_summary + "\n")
    fh.write(test_report)
print(f"Updated {report_path}")
summary_df.round(4)